# Feature engineering

## 0. Configs

### 0.1 Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import warnings

# configurações para importar as funcões do módulo utils
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# funções utils
from src.utils_eda import months, days, faixa_etaria

# configurações de exibição
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

### 0.2 Base de dados

In [3]:
# ler tabela
df_raw = pd.read_csv('../data/raw/bank-marketing-data-set/bank-additional-full.csv', sep = ';')

print(f"Tamanho do dataset: {df_raw.shape}")
df_raw.head()

Tamanho do dataset: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 1. Feature engineering

In [5]:
# 1. inicializar tabela
df = df_raw.copy()

# 2. tratar nomes dos meses e dias da semana
df['month'] = df['month'].map(months)
df['day_of_week'] = df['day_of_week'].map(days)

# 3. criar coluna de ano
df['month_num'] = df['month'].str[:2].astype(int)
virada_ano = df['month_num'] < df['month_num'].shift()
df['year'] = 2008 + virada_ano.cumsum()
df = df.drop(columns='month_num')

# 4. criar coluna de faixa etária
df['faixa_etaria'] = df['age'].apply(faixa_etaria)

# 5. substituir . por _ em nomes de colunas
df.columns = [c.replace('.', '_') for c in df.columns]

# 6. selecionar colunas
df = df[[
    # perfil
    'age', 'job', 'marital', 'education', 'faixa_etaria',

    # situação financeira
    'housing', 'loan', 

    # dados dos contatos
    'contact', 'month', 'year', 'day_of_week',

    # histórico de relacionamento
    'previous', 'poutcome',

    # indicadores econômicos
    'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed',

    # target
    'y'
]]

## removidas:
# - campaign; não faz sentido usar a quantidade de contatos porque isso não necessariamente será usado ao fazer o predict
# - duration: não tem como saber qual será a duração da ligação ao fazer o contato
# - pdays: a maioria é 999 (equivalente a null)
# - default: se tem crédito por padrão, apenas 3 estão como yes, o resto é no ou unknown

print(f"Tamanho da tabela tratada: {df_raw.shape}")
df.head()

Tamanho da tabela tratada: (41188, 21)


,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,56,housemaid,married,basic.4y,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,Entre 31 e 40 anos,yes,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,Entre 31 e 40 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,Entre 50 e 60 anos,no,yes,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [30]:
df.dtypes

age                 int64
job                   str
marital               str
education             str
faixa_etaria          str
housing               str
loan                  str
contact               str
month                 str
year                int64
day_of_week           str
previous            int64
poutcome              str
emp_var_rate      float64
cons_price_idx    float64
cons_conf_idx     float64
euribor3m         float64
nr_employed       float64
y                     str
dtype: object

## 2. Salvar tabela trusted

In [31]:
df.to_parquet('../data/trusted/tabela_analitica.parquet', engine = 'pyarrow')

## 3. Salvar tabelas de dados econômicos

In [10]:
df.head()

,age,job,marital,education,faixa_etaria,default,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,56,housemaid,married,basic.4y,Entre 50 e 60 anos,no,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,Entre 50 e 60 anos,unknown,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,Entre 31 e 40 anos,no,yes,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,Entre 31 e 40 anos,no,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,Entre 50 e 60 anos,no,no,yes,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [15]:
ind_economicos = ['emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'nr_employed']

for c in ind_economicos:
    tmp = df[['month', 'year', c]].drop_duplicates().reset_index(drop = True)
    tmp.to_parquet(f'../data/trusted/{c}_mensal.parquet', engine = 'pyarrow')

In [8]:
euribor = df[['month', 'year', 'euribor3m']].drop_duplicates().reset_index(drop = True)
euribor['month_position'] = euribor.groupby(['month', 'year']).cumcount()
euribor = euribor.merge(euribor.groupby(['month', 'year'])['month_position'].max().reset_index().rename(columns = {'month_position':'month_qtd'}))
euribor['month_position_num'] = euribor['month_position'] / euribor['month_qtd']
euribor['month_position'] = np.where(euribor['month_position_num'] < (1/3), 'início', np.where(euribor['month_position_num'] <= (2/3), 'meio', 'fim'))
euribor = euribor.groupby(['month', 'year', 'month_position']).agg({'euribor3m':'mean', 'month_qtd':'mean'}).reset_index()
euribor = euribor.drop(columns = 'month_qtd')

# display(euribor)
euribor.sort_values(['year', 'month']).to_parquet(f'../data/trusted/euribor3m_mensal.parquet', engine = 'pyarrow')